# EDA — Open-Meteo

Análise das séries diárias de **2025 para as 27 capitais**. A Bronze preserva listas dentro da resposta da API; nesta análise, criamos uma visão temporária com uma linha por capital e dia.

Investigamos estrutura, cobertura, ausências, duplicações, unidades, consistência física e variação ao longo do ano. Os arquivos da Bronze permanecem intactos. Um único ano não estabelece uma normal climatológica; os números também não representam todo o território brasileiro.

## 1. Bibliotecas e caminhos

Execute com o kernel do ambiente virtual do projeto. Apenas tabelas pequenas e resultados agregados seguem para os gráficos.

In [ ]:
import os
import sys
from datetime import date
from pathlib import Path

import matplotlib.pyplot as plt
from pyspark.sql import SparkSession
from pyspark.sql import functions as F


project_directory = Path.cwd().resolve()

while not (project_directory / "pipeline" / "bronze").is_dir() or not (project_directory / "data").is_dir():

    if project_directory == project_directory.parent:
        raise FileNotFoundError("Execute o notebook dentro do projeto.")

    project_directory = project_directory.parent

YEAR = 2025
START_DATE = date(YEAR, 1, 1)
END_DATE = date(YEAR, 12, 31)
EXPECTED_DAYS = (END_DATE - START_DATE).days + 1
BRONZE_DIRECTORY = project_directory / "data" / "bronze"
WEATHER_DIRECTORY = BRONZE_DIRECTORY / "open_meteo" / "weather" / str(YEAR)
MUNICIPALITIES_DIRECTORY = BRONZE_DIRECTORY / "ibge" / "municipalities"
local_filesystem_jar = project_directory / "pipeline" / "bronze" / "lib" / "hadoop-bare-naked-local-fs-0.1.0.jar"

for required_path in [WEATHER_DIRECTORY, MUNICIPALITIES_DIRECTORY, local_filesystem_jar]:

    if not required_path.exists():
        raise FileNotFoundError(f"Caminho necessário não encontrado: {required_path}")

print(f"Período esperado: {START_DATE} a {END_DATE} ({EXPECTED_DAYS} dias)")


## 2. Sessão Spark

Usamos a configuração local já validada na Bronze. As datas diárias serão interpretadas como datas do calendário local indicado em cada resposta, sem conversão para instantes UTC.

In [ ]:
os.environ["PYSPARK_PYTHON"] = sys.executable

spark = (
    SparkSession.builder
    .appName("Viajando com o dado - EDA Open-Meteo")
    .master("local[2]")
    .config("spark.sql.shuffle.partitions", "2")
    .config("spark.pyspark.python", sys.executable)
    .config("spark.driver.extraClassPath", str(local_filesystem_jar))
    .config("spark.executor.extraClassPath", str(local_filesystem_jar))
    .config("spark.hadoop.fs.file.impl", "com.globalmentor.apache.hadoop.fs.BareLocalFileSystem")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")
print(f"Versão do Spark: {spark.version}")


## 3. Leitura e estrutura

Cada registro contém os metadados de uma capital, as unidades e as listas diárias. A referência do IBGE também está limitada às capitais selecionadas na coleta.

In [ ]:
weather = (
    spark.read
    .option("recursiveFileLookup", True)
    .option("pathGlobFilter", "*.parquet")
    .parquet(WEATHER_DIRECTORY.resolve().as_posix())
    .cache()
)

capitals = (
    spark.read
    .option("recursiveFileLookup", True)
    .option("pathGlobFilter", "*.parquet")
    .parquet(MUNICIPALITIES_DIRECTORY.resolve().as_posix())
    .select(F.col("id").alias("municipality_id"), F.col("nome").alias("reference_name"), F.col("state_code").alias("reference_state"))
    .distinct()
)

source_count = weather.count()
reference_count = capitals.count()
print(f"Respostas: {source_count}; capitais na referência: {reference_count}")

weather.printSchema()
weather.select("municipality_id", "name", "state_code", "start_date", "end_date", "weather.timezone", "weather.daily_units").orderBy("state_code").show(27, truncate=False)


## 4. Identidade, período e unidades

Comparamos ID, nome e UF com o IBGE. Diferenças de unidade são alertadas, sem conversão automática. Não exigimos o mesmo fuso para todas as capitais. As coordenadas retornadas pela fonte não são uma validação de limites municipais.

In [ ]:
variables = ["temperature_2m_mean", "temperature_2m_min", "temperature_2m_max", "precipitation_sum", "wind_speed_10m_max"]
expected_units = {
    "time": "iso8601",
    "temperature_2m_mean": "°C",
    "temperature_2m_min": "°C",
    "temperature_2m_max": "°C",
    "precipitation_sum": "mm",
    "wind_speed_10m_max": "km/h",
}

identity = weather.join(capitals, "municipality_id", "full")
identity_issues = identity.filter(
    F.col("name").isNull()
    | F.col("reference_name").isNull()
    | ~F.col("name").eqNullSafe(F.col("reference_name"))
    | ~F.col("state_code").eqNullSafe(F.col("reference_state"))
)

duplicate_sources = weather.groupBy("municipality_id").count().filter(F.col("count") > 1)
duplicate_states = weather.groupBy("state_code").count().filter(F.col("count") > 1)

period_issues = weather.filter(
    ~F.col("start_date").eqNullSafe(F.lit(START_DATE.isoformat()))
    | ~F.col("end_date").eqNullSafe(F.lit(END_DATE.isoformat()))
)

unit_condition = F.lit(False)

for variable, unit in expected_units.items():
    unit_condition = unit_condition | ~F.col("weather.daily_units")[variable].eqNullSafe(F.lit(unit))

unit_issues = weather.filter(unit_condition)
metadata_issues = weather.filter(
    F.col("weather.timezone").isNull()
    | (F.trim(F.col("weather.timezone")) == "")
    | F.col("weather.utc_offset_seconds").isNull()
    | F.col("weather.latitude").isNull()
    | F.isnan("weather.latitude")
    | ~F.col("weather.latitude").between(-90, 90)
    | F.col("weather.longitude").isNull()
    | F.isnan("weather.longitude")
    | ~F.col("weather.longitude").between(-180, 180)
)

checks = {}

for label, table in {
    "Identidade divergente ou ausente": identity_issues,
    "IDs com respostas duplicadas": duplicate_sources,
    "UFs com respostas duplicadas": duplicate_states,
    "Período declarado divergente": period_issues,
    "Unidades divergentes ou ausentes": unit_issues,
    "Metadados geográficos ou de fuso inválidos": metadata_issues,
}.items():
    checks[label] = table.count()
    print(f"{label}: {checks[label]}")

identity_issues.select("municipality_id", "name", "state_code", "reference_name", "reference_state").show(truncate=False)
unit_issues.select("state_code", "weather.daily_units").show(truncate=False)


## 5. Integridade das listas

Cada lista deve conter o número esperado de dias. Detectamos listas ausentes ou de tamanho diferente antes de expandir. `arrays_zip` alinha pela posição e preenche diferenças com nulos, preservando valores excedentes para inspeção.

In [ ]:
series = weather.select(
    "municipality_id", "name", "state_code", "source_file",
    F.col("weather.timezone").alias("timezone"),
    F.col("weather.daily.time").alias("dates"),
    "weather.daily.temperature_2m_mean", "weather.daily.temperature_2m_min",
    "weather.daily.temperature_2m_max", "weather.daily.precipitation_sum",
    "weather.daily.wind_speed_10m_max",
)

array_names = ["dates"] + variables
length_columns = [F.col("municipality_id"), F.col("state_code")]
array_condition = F.lit(False)

for variable in array_names:
    length_columns.append(F.size(variable).alias(variable + "_length"))
    array_condition = array_condition | F.col(variable).isNull() | (F.size(variable) != EXPECTED_DAYS)

array_issues = series.filter(array_condition)
checks["Respostas com listas ausentes ou incompletas"] = array_issues.count()
series.select(*length_columns).orderBy("state_code").show(27, truncate=False)
print(f"Respostas com problemas nas listas: {checks['Respostas com listas ausentes ou incompletas']}")

expansion = series

for variable in array_names:
    array_type = "array<string>" if variable == "dates" else "array<double>"
    expansion = expansion.withColumn(variable, F.coalesce(F.col(variable), F.array().cast(array_type)))

expanded = expansion.select(
    "municipality_id", "name", "state_code", "source_file", "timezone",
    F.posexplode_outer(F.arrays_zip(*array_names)).alias("day_index", "daily_record"),
)

# Remove apenas o marcador criado quando todas as listas estão vazias.
daily = (
    expanded
    .filter(F.col("day_index").isNotNull())
    .select("municipality_id", "name", "state_code", "source_file", "timezone", "day_index", "daily_record.*")
    .withColumnRenamed("dates", "date_raw")
    .withColumn("date", F.expr("try_cast(date_raw AS DATE)"))
    .cache()
)

daily_count = daily.count()

print(f"Registros diários: {daily_count}")
daily.orderBy("state_code", "day_index").show(5, truncate=False)


## 6. Datas, duplicações e cobertura

O calendário esperado é construído independentemente das respostas, com todas as capitais do IBGE e todos os dias do ano. Assim, uma capital ausente por inteiro também aparece como falta de cobertura. Não confundimos presença da data com disponibilidade das medições.

In [ ]:
invalid_dates = daily.filter(
    F.col("date").isNull()
    | ~F.col("date_raw").rlike(r"^\d{4}-\d{2}-\d{2}$")
)
outside_period = daily.filter(F.col("date").isNotNull() & ~F.col("date").between(START_DATE, END_DATE))
duplicate_days = daily.groupBy("municipality_id", "date").count().filter(F.col("count") > 1)

calendar = spark.range(1).select(
    F.explode(F.sequence(F.lit(START_DATE), F.lit(END_DATE))).alias("date")
)
expected_calendar = capitals.select("municipality_id", "reference_state").crossJoin(calendar)
missing_days = expected_calendar.join(daily.select("municipality_id", "date").distinct(), ["municipality_id", "date"], "left_anti")
unexpected_days = daily.join(expected_calendar.select("municipality_id", "date"), ["municipality_id", "date"], "left_anti")

for label, table in {
    "Datas inválidas ou fora do formato ISO": invalid_dates,
    "Datas fora do período": outside_period,
    "Chaves capital/data duplicadas": duplicate_days,
    "Dias esperados ausentes": missing_days,
    "Registros fora do calendário esperado": unexpected_days,
}.items():
    checks[label] = table.count()
    print(f"{label}: {checks[label]}")

daily.groupBy("state_code").agg(
    F.count("*").alias("rows"),
    F.countDistinct("date").alias("distinct_days"),
    F.min("date").alias("first_date"),
    F.max("date").alias("last_date"),
).orderBy("state_code").show(27)
missing_days.show(10, truncate=False)


## 7. Ausências e valores não finitos

Nulos, NaN e infinito são contados separadamente. Precipitação igual a zero representa um dia sem chuva estimada e continua válida. A visão numérica usada nos resumos substitui apenas valores não finitos por nulo, sem alterar a Bronze.

In [ ]:
availability_rows = []
numeric_daily = daily

for variable in variables:
    value = F.col(variable)

    nonfinite = F.isnan(value) | (F.abs(value) == F.lit(float("inf")))
    counts = daily.agg(
        F.sum(F.when(value.isNull(), 1).otherwise(0)).alias("missing"),
        F.sum(F.when(value.isNotNull() & nonfinite, 1).otherwise(0)).alias("nonfinite"),
    ).first()

    availability_rows.append((variable, int(counts["missing"] or 0), int(counts["nonfinite"] or 0)))
    numeric_daily = numeric_daily.withColumn(variable, F.when(~nonfinite, value))

availability = spark.createDataFrame(availability_rows, "variable string, missing long, nonfinite long")
availability.show(truncate=False)

# Os resumos temporais usam apenas datas ISO válidas no período esperado.
numeric_daily = numeric_daily.filter(
    F.col("date").between(START_DATE, END_DATE)
    & F.col("date_raw").rlike(r"^\d{4}-\d{2}-\d{2}$")
)
numeric_daily.select(*variables).summary("count", "mean", "stddev", "min", "25%", "50%", "75%", "max").show(truncate=False)


## 8. Consistência física

Verificamos chuva e vento negativos e a ordem entre temperaturas mínima, média e máxima. Temperaturas negativas são permitidas. Cada regra exige os valores necessários; nulos já foram contabilizados acima. Os alertas não provocam exclusão automática.

In [ ]:
domain_rules = {
    "Chuva negativa": F.col("precipitation_sum") < 0,
    "Vento negativo": F.col("wind_speed_10m_max") < 0,
    "Temperatura mínima maior que a máxima": F.col("temperature_2m_min") > F.col("temperature_2m_max"),
    "Temperatura média abaixo da mínima": F.col("temperature_2m_mean") < F.col("temperature_2m_min"),
    "Temperatura média acima da máxima": F.col("temperature_2m_mean") > F.col("temperature_2m_max"),
}
domain_condition = F.lit(False)

for label, condition in domain_rules.items():
    checks[label] = numeric_daily.filter(condition).count()
    domain_condition = domain_condition | F.coalesce(condition, F.lit(False))
    print(f"{label}: {checks[label]}")

domain_issues = numeric_daily.filter(domain_condition)
domain_issue_count = domain_issues.count()
domain_issues.select("state_code", "date", *variables).show(20, truncate=False)


## 9. Distribuições e valores extremos

O IQR sinaliza temperaturas médias diárias distantes da distribuição de cada capital e mês. Esse critério exploratório não comprova erro; eventos meteorológicos extremos precisam ser preservados e investigados. A comparação por mês reduz a mistura de estações do ano.

In [ ]:
monthly_values = numeric_daily.withColumn("month", F.month("date"))

quartiles = monthly_values.groupBy("municipality_id", "month").agg(
    F.percentile_approx("temperature_2m_mean", [0.25, 0.75], 10000).alias("quartiles")
)

temperature_flags = (
    monthly_values
    .join(quartiles, ["municipality_id", "month"])
    .withColumn("q1", F.col("quartiles")[0])
    .withColumn("q3", F.col("quartiles")[1])
    .withColumn("iqr", F.col("q3") - F.col("q1"))
    .filter(
        (F.col("temperature_2m_mean") < F.col("q1") - 1.5 * F.col("iqr"))
        | (F.col("temperature_2m_mean") > F.col("q3") + 1.5 * F.col("iqr"))
    )
)

iqr_count = temperature_flags.count()
print(f"Alertas de temperatura pelo IQR de capital/mês: {iqr_count}")

temperature_flags.select("state_code", "date", "temperature_2m_mean", "q1", "q3").orderBy("state_code", "date").show(20)


## 10. Temperatura ao longo do ano

Cada célula mostra a média das temperaturas médias diárias disponíveis. A tabela informa quantos dias contribuíram: médias com cobertura parcial precisam de cautela. O gráfico representa somente 2025.

In [ ]:
monthly_summary = monthly_values.groupBy("state_code", "month").agg(
    F.avg("temperature_2m_mean").alias("mean_temperature"),
    F.count("temperature_2m_mean").alias("temperature_days"),
    F.countDistinct("date").alias("distinct_days"),
    F.count("*").alias("rows"),
    F.dayofmonth(F.last_day(F.min("date"))).alias("expected_days"),
)

monthly_summary.orderBy("state_code", "month").show(36)

state_codes = []

for row in capitals.select("reference_state").distinct().orderBy("reference_state").collect():
    state_codes.append(row["reference_state"])

temperature_matrix = []

for state_code in state_codes:
    temperature_matrix.append([float("nan")] * 12)

for row in monthly_summary.collect():

    if row["state_code"] in state_codes and row["mean_temperature"] is not None:
        state_index = state_codes.index(row["state_code"])
        temperature_matrix[state_index][row["month"] - 1] = row["mean_temperature"]

figure, axis = plt.subplots(figsize=(12, 10))
image = axis.imshow(temperature_matrix, aspect="auto", cmap="YlOrRd")
axis.set_yticks(range(len(state_codes)), state_codes)
axis.set_xticks(range(12), ["Jan", "Fev", "Mar", "Abr", "Mai", "Jun", "Jul", "Ago", "Set", "Out", "Nov", "Dez"])
axis.set_title(f"Temperatura média diária agregada por mês — capitais, {YEAR}")
axis.set_xlabel("Mês")
axis.set_ylabel("UF da capital")

figure.colorbar(image, ax=axis, label="Temperatura média (°C)")
figure.tight_layout()

plt.show()
plt.close(figure)


## 11. Precipitação acumulada por capital

Só apresentamos total anual para séries com todos os dias esperados, sem duplicação e com precipitação finita e não negativa em cada dia. Uma soma parcial não será apresentada como total anual. O resultado descreve a estimativa da fonte para cada capital.

In [ ]:
annual_rain = numeric_daily.groupBy("municipality_id", "name", "state_code").agg(
    F.count("*").alias("rows"),
    F.countDistinct("date").alias("distinct_days"),
    F.count("precipitation_sum").alias("rain_days"),
    F.sum(F.when(F.col("precipitation_sum") < 0, 1).otherwise(0)).alias("negative_days"),
    F.sum("precipitation_sum").alias("rain_sum"),
)

annual_rain = annual_rain.withColumn(
    "complete",
    (F.col("rows") == EXPECTED_DAYS)
    & (F.col("distinct_days") == EXPECTED_DAYS)
    & (F.col("rain_days") == EXPECTED_DAYS)
    & (F.col("negative_days") == 0),
)

annual_rain.orderBy("state_code").show(27)

rain_labels = []
rain_totals = []

for row in annual_rain.filter("complete").orderBy("rain_sum").collect():
    rain_labels.append(f"{row['name']} ({row['state_code']})")
    rain_totals.append(row["rain_sum"])

figure, axis = plt.subplots(figsize=(12, 10))
axis.barh(rain_labels, rain_totals, color="steelblue")
axis.set_title(f"Precipitação acumulada — capitais com série completa, {YEAR}")
axis.set_xlabel("Precipitação anual (mm)")
axis.grid(axis="x", alpha=0.25)
figure.tight_layout()

plt.show()
plt.close(figure)


## 12. Inspeção de uma capital

São Paulo ilustra a relação visual entre mínima, média, máxima e precipitação. As datas seguem o calendário local da resposta. Valores ausentes aparecem como lacunas, sem preenchimento com zero. O gráfico não estabelece relação causal entre chuva e temperatura.

In [ ]:
SELECTED_STATE = "SP"
selected_daily = numeric_daily.filter(F.col("state_code") == SELECTED_STATE).orderBy("date", "day_index")
plot_dates = []
plot_minimum = []
plot_mean = []
plot_maximum = []
plot_rain = []

for row in selected_daily.select("date", *variables).collect():
    plot_dates.append(row["date"])

    for variable, values in {
        "temperature_2m_min": plot_minimum,
        "temperature_2m_mean": plot_mean,
        "temperature_2m_max": plot_maximum,
        "precipitation_sum": plot_rain,
    }.items():
        value = row[variable]
        values.append(float("nan") if value is None else value)

figure, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
axes[0].fill_between(plot_dates, plot_minimum, plot_maximum, alpha=0.25, label="Mínima a máxima")
axes[0].plot(plot_dates, plot_mean, linewidth=1, label="Média diária")
axes[0].set_ylabel("Temperatura (°C)")
axes[0].set_title(f"Série diária da capital de {SELECTED_STATE} — {YEAR}")
axes[0].legend()
axes[1].bar(plot_dates, plot_rain, color="steelblue", width=1)
axes[1].set_ylabel("Precipitação (mm)")
axes[1].set_xlabel("Data local")

figure.tight_layout()

plt.show()
plt.close(figure)


## 13. Conclusões e limites

O resumo abaixo é calculado a cada execução. Alertas de IQR não equivalem a dados incorretos. A disponibilidade de um valor também não garante sua precisão meteorológica. Antes de comparar com INMET, será necessário compatibilizar ponto de referência, unidades, período e definição do dia. Não fazemos essa comparação nesta EDA.

In [ ]:
print(f"Respostas analisadas: {source_count}; referência IBGE: {reference_count} capitais.")
print(f"Registros diários: {daily_count}; esperado na referência: {reference_count * EXPECTED_DAYS}.")

for label, count in checks.items():
    print(f"{label}: {count}")

missing_total = 0
nonfinite_total = 0

for variable, missing, nonfinite in availability_rows:
    missing_total += missing
    nonfinite_total += nonfinite

print(f"Células numéricas ausentes: {missing_total}; não finitas: {nonfinite_total}.")
print(f"Dias com algum alerta físico: {domain_issue_count}.")
print(f"Alertas de IQR preservados para investigação: {iqr_count}.")
print(f"Capitais com precipitação anual completa: {annual_rain.filter('complete').count()}.")
print("As transformações desta análise existem apenas em memória; a Bronze foi preservada.")


## 14. Encerramento

Para continuar explorando os DataFrames, execute esta célula somente ao terminar.

In [ ]:
daily.unpersist()
weather.unpersist()
spark.stop()
